# Benchmark DARU — cascata System 1 → System 2 no mesmo modelo (PR5)

Tese ([SPEC-DARU.md](https://github.com/Grazinascito/adaptive-cascade-routing/blob/benchmark-daru/SPEC-DARU.md)): a economia de custo da arquitetura híbrida do iFood (Sugar) também existe quando trocamos "modelo pequeno vs modelo grande" por "System 1 vs System 2 no mesmo modelo" (`Qwen/Qwen3-4B-AWQ`, servidor vLLM único):

- **SC-A — System 1 sozinho**: decisão de 1 token no `/v1/systemone` (rápido e barato).
- **SC-B — System 2 sozinho**: CoT + JSON estrito no `/v1/chat/completions` (lento e caro).
- **SC-C — cascata**: System 1 primeiro; se P_top < α, o item sobe para o System 2 (regra R_α do RouteLLM).

Dataset: DARU (18 maiores varejistas do Brasil) → 12 categorias de comida/bebida, validação 200 + teste congelado 600, cauda = descrição com ≤ 20 caracteres.

Pipeline deste notebook: checagem de GPU → dependências → clone do repo (`benchmark-daru`) → dataset via kagglehub → servidor vLLM → **pilot** (gate de 100 itens; o notebook para aqui se reprovar) → run completo (validação + sweep de α + teste congelado) → gráficos → download dos artefatos.

> Verificação: 86 testes + ruff + mypy verdes no CI do [PR #7](https://github.com/Grazinascito/adaptive-cascade-routing/pull/7). Este notebook executa apenas componentes reais — servidor vLLM local e dataset DARU real; nenhuma etapa usa dados ou respostas simuladas.


In [ ]:
import subprocess

gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0 or "T4" not in (gpu.stdout + gpu.stderr):
    raise RuntimeError(
        "Sem GPU T4 disponível. Em 'Ambiente de execução > Alterar o tipo do "
        "ambiente de execução', selecione GPU T4 e rode o notebook de novo."
    )
print("GPU check: T4 present")

In [ ]:
%pip install -q vllm httpx pyyaml pandas kagglehub openpyxl matplotlib

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Grazinascito/adaptive-cascade-routing.git"
BRANCH = "benchmark-daru"
REPO_DIR = Path("adaptive-cascade-routing").resolve()

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
rev = subprocess.run(
    ["git", "log", "--oneline", "-1"], cwd=REPO_DIR, capture_output=True, text=True
)
print(rev.stdout)

In [ ]:
import glob

import kagglehub

dataset_root = kagglehub.dataset_download(
    "gilsileydaru/classificao-produtos-varejo-cpg-prbr"
)
xlsx_matches = glob.glob(
    f"{dataset_root}/**/DATASETPRODUTOSVAREJOBRASIL.xlsx", recursive=True
)
if not xlsx_matches:
    raise FileNotFoundError(f"xlsx do DARU não encontrado em {dataset_root}")
print("dataset cached at:", xlsx_matches[0])

## Servidor vLLM — Qwen3-4B-AWQ

`src.engine.server` sobe o servidor como subprocesso com prefix caching e 90% da VRAM (≤ 14,4 GB dos 16 GB da T4). O health check é `GET /v1/models` até responder 200; na primeira execução o vLLM baixa ~2,7 GB de pesos — por isso o timeout de startup de 1800 s. Se algo falhar, o log completo fica em `vllm_server.log`.

In [ ]:
import sys

import httpx

sys.path.insert(0, str(REPO_DIR))

from src.benchmark.runner import load_config
from src.engine import server

config = load_config(REPO_DIR / "config" / "benchmark_config.yaml")
port = config["serving"]["port"]
base_url = f"http://127.0.0.1:{port}"

print("server argv:", " ".join(server.build_server_command(config["model"]["id"], port)))
print("starting vLLM (first AWQ load on T4 takes a few minutes)...")
server.start_server(config["model"]["id"], port=port, startup_timeout_s=1800.0)
print("health:", server.health_check(base_url))
print("modelos:", httpx.get(f"{base_url}/v1/models", timeout=10).json())

## Pilot — gate dos 100 itens (SPEC-DARU §6)

100 itens de validação (70 cabeça + 30 cauda), System 1 e System 2 medidos uma vez. O gate: a acurácia do System 2 na cauda precisa superar a do System 1 em **≥ 10 pontos percentuais** — é a falha do ASSIN 2 que queremos descobrir em 1 hora, não depois do run completo de GPU. Se o gate reprovar, a célula **interrompe o notebook**.

In [ ]:
import subprocess
import sys

pilot = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.benchmark.run",
        "--config",
        "config/benchmark_config.yaml",
        "--pilot",
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
print(pilot.stdout)
if pilot.returncode != 0:
    print(pilot.stderr)
    raise RuntimeError(
        "PILOT REPROVADO: o System 2 não superou o System 1 na cauda em >= 10 pp "
        "(SPEC-DARU §6). O notebook para aqui — diagnóstico honesto em vez de "
        "run completo inválido."
    )
print("PILOT GATE PASSED — proceeding to full run.")

## Run completo — validação + sweep de α + teste congelado

1. **Validação (200 itens)**: System 1 e System 2 medidos uma única vez; o sweep de α (0,70–0,90) é aritmética pura sobre as saídas salvas. Melhor α = maior acurácia da cascata; empate decide pelo menor custo de tokens.
2. **Teste congelado (600 itens, medido 1x)**: SC-A, SC-B e SC-C com o α escolhido, com warmup de 5 requisições por engine antes de cada cenário.

Artefato bruto: `results/raw_benchmark_data.json` (meta, cenários por item, sweep, sumários e os três gates: gap de cauda ≥ 10 pp, APGR ≥ 0,5, tokens ≤ 40% do System 2).

In [ ]:
import json
import subprocess
import sys

full = subprocess.run(
    [sys.executable, "-m", "src.benchmark.run", "--config", "config/benchmark_config.yaml"],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
print(full.stdout)
if full.returncode != 0:
    print(full.stderr)
    raise RuntimeError("run completo falhou; veja a saída acima.")

RESULTS = json.loads(
    (REPO_DIR / "results" / "raw_benchmark_data.json").read_text(encoding="utf-8")
)
print("result keys:", list(RESULTS))
for name, summary in RESULTS["summaries"].items():
    if name != "VALIDATION":
        keys = ("n_items", "accuracy", "accuracy_tail",
                "routed_to_s2_count", "decode_tokens_total")
        print(name, {k: summary[k] for k in keys})

## Gráficos

1. **Fronteira custo × acurácia**: tokens de saída somados vs acurácia por cenário — a tese se sustenta se o SC-C ficar perto do SC-B em acurácia com uma fração dos tokens.
2. **Curva do sweep de α** na validação: acurácia da cascata e taxa de roteação vs α (quanto maior o α, mais itens sobem para o System 2).

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

scenarios = {n: s for n, s in RESULTS["summaries"].items() if n != "VALIDATION"}

fig, ax = plt.subplots(figsize=(7, 5))
for name, summary in scenarios.items():
    ax.scatter(summary["decode_tokens_total"], summary["accuracy"], s=120)
    ax.annotate(
        name,
        (summary["decode_tokens_total"], summary["accuracy"]),
        xytext=(6, 4),
        textcoords="offset points",
    )
ax.set_xlabel("tokens de saída somados (custo)")
ax.set_ylabel("acurácia")
ax.set_title("Fronteira custo × acurácia por cenário (teste congelado)")
ax.grid(True, alpha=0.3)
fig.savefig(REPO_DIR / "results" / "custo_acuracia_fronteira.png", dpi=150, bbox_inches="tight")
print("saved: results/cost_accuracy_frontier.png")

sweep = RESULTS["alpha_sweep"]["alphas"]
alphas = sorted(sweep, key=float)
accuracies = [sweep[a]["accuracy"] for a in alphas]
routed = [sweep[a]["routed_rate"] for a in alphas]

fig, ax1 = plt.subplots(figsize=(7, 5))
ax1.plot(alphas, accuracies, "o-", label="acurácia da cascata")
ax1.set_xlabel("α (limiar de P_top)")
ax1.set_ylabel("acurácia")
ax2 = ax1.twinx()
ax2.plot(alphas, routed, "s--", color="tab:orange", label="taxa de roteação")
ax2.set_ylabel("taxa de roteação (fração que sobe ao S2)")
ax1.set_title(f"Sweep de α na validação (melhor α = {RESULTS['alpha_sweep']['best_alpha']})")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="center left")
ax2.legend(loc="lower right")
fig.savefig(REPO_DIR / "results" / "alpha_sweep.png", dpi=150, bbox_inches="tight")
print("saved: results/alpha_sweep.png")
plt.show()

In [ ]:
import shutil

zip_path = shutil.make_archive(
    "results_daru_benchmark", "zip", root_dir=REPO_DIR / "results"
)
print("archive:", zip_path)
print("contents:", sorted(p.name for p in (REPO_DIR / "results").iterdir()))

try:
    from google.colab import files

    files.download(zip_path)
except ImportError:
    print("outside Colab: download the archive from the path above.")